# Olist – Tiền xử lý dữ liệu (ETL sang Snowflake Schema) — bản chạy trên Kaggle

Notebook này chuyển dữ liệu thô (đọc trực tiếp từ file **SQLite** `olist.sqlite`) thành các bảng Fact/Dimension đúng theo thiết kế `olist_snowflake_schema.drawio`.

**Cách dùng trên Kaggle:** vào **Add Input** → chọn dataset chứa `olist.sqlite` (hoặc `.db`) → chạy notebook, không cần chỉnh gì thêm — notebook tự tìm file trong `/kaggle/input/`.

**Fact:**
- `FACT_ORDER_ITEMS` — grain: 1 dòng / order item
- `FACT_ORDER_PAYMENT` — grain: 1 dòng / lần thanh toán (order_id + payment_sequential)
- `FACT_SELLER_LEAD` — grain: 1 dòng / lead (gộp leads_qualified + leads_closed)

**Dimension:**
- `DIM_CUSTOMER`, `DIM_SELLER`, `DIM_PRODUCT`, `DIM_PRODUCT_CATEGORY`, `DIM_LOCATION`, `DIM_REVIEW`

Các vấn đề chất lượng dữ liệu đã xử lý (phát hiện trong quá trình phân tích):
1. `geolocation_zip_code_prefix` **không unique** (~52 dòng/zip) → `DIM_LOCATION` giữ nguyên toàn bộ dòng gốc (không aggregate), khớp FK bằng cả `(zip, city, state)` để giảm ambiguity.
2. `product_category_name` có thể không tồn tại trong `product_category_name_translation` → bổ sung dòng thiếu, không dùng INNER JOIN.
3. `order_payments` và `order_reviews` có N dòng / 1 order (fan-out) → giữ làm bảng/fact riêng, không gộp thẳng vào `FACT_ORDER_ITEMS`.
4. Cột zip trong SQLite được lưu kiểu **INTEGER** → dễ mất số 0 ở đầu → luôn ép lại dạng chuỗi có đệm 0 (zfill) sau khi đọc.
5. `leads_closed.seller_id` có ~55% không tồn tại trong `sellers` (seller mới, chưa có đơn hàng) → vẫn giữ FK, chấp nhận orphan, đánh dấu bằng cờ `is_won`.


## 1. Import thư viện

In [ ]:
import sqlite3
import glob
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


## 2. Kết nối file SQLite trên Kaggle

Kaggle nạp dataset qua **Add Input**, dữ liệu nằm dưới `/kaggle/input/<ten-dataset>/...`.
**Sửa `SQLITE_PATH` bên dưới thành đường dẫn thật** tới file `.sqlite` sau khi Add Input xong
(xem trong panel Input bên phải, hoặc chạy `!find /kaggle/input -name "*.sqlite"` ở 1 cell riêng để lấy đường dẫn).

In [ ]:
SQLITE_PATH = Path('/kaggle/input/ten-dataset/olist.sqlite')   # <-- doi thanh duong dan that tren Kaggle

IN_KAGGLE = Path('/kaggle/input').exists()
OUT_DIR = Path('/kaggle/working/processed') if IN_KAGGLE else Path('processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert SQLITE_PATH.exists(), f'Khong tim thay {SQLITE_PATH} — kiem tra lai duong dan sau khi Add Input'

conn = sqlite3.connect(SQLITE_PATH)
print('SQLITE_PATH =', SQLITE_PATH.resolve())
print('OUT_DIR     =', OUT_DIR.resolve())
print('Cac bang co trong sqlite:')
print(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn))


## 3. Đọc dữ liệu thô (query trực tiếp từ SQLite)

In [ ]:
customers   = pd.read_sql_query('SELECT * FROM customers', conn)
geolocation = pd.read_sql_query('SELECT * FROM geolocation', conn)
leads_closed    = pd.read_sql_query('SELECT * FROM leads_closed', conn)
leads_qualified = pd.read_sql_query('SELECT * FROM leads_qualified', conn)
order_items    = pd.read_sql_query('SELECT * FROM order_items', conn)
order_payments = pd.read_sql_query('SELECT * FROM order_payments', conn)
order_reviews  = pd.read_sql_query('SELECT * FROM order_reviews', conn)
orders         = pd.read_sql_query('SELECT * FROM orders', conn)
category_translation = pd.read_sql_query('SELECT * FROM product_category_name_translation', conn)
products = pd.read_sql_query('SELECT * FROM products', conn)
sellers  = pd.read_sql_query('SELECT * FROM sellers', conn)

conn.close()

tables = {
    'customers': customers, 'geolocation': geolocation, 'leads_closed': leads_closed,
    'leads_qualified': leads_qualified, 'order_items': order_items, 'order_payments': order_payments,
    'order_reviews': order_reviews, 'orders': orders, 'category_translation': category_translation,
    'products': products, 'sellers': sellers,
}
for name, df in tables.items():
    print(f'{name:22s} shape={df.shape}')


## 4. Kiểm tra chất lượng dữ liệu (data quality checks)

Xác nhận lại các vấn đề đã phát hiện trước khi xử lý, để notebook tự chứng minh được lý do các bước biến đổi bên dưới.

In [ ]:
# 4.1 Zip code trong geolocation KHONG unique
dup_zip = geolocation['geolocation_zip_code_prefix'].value_counts()
print('So zip khac nhau trong geolocation:', dup_zip.shape[0])
print('So dong geolocation:', len(geolocation))
print('Trung binh so dong / zip:', round(len(geolocation) / dup_zip.shape[0], 2))
assert dup_zip.shape[0] < len(geolocation), 'geolocation_zip_code_prefix khong unique -> phai xu ly khi lam FK'


In [ ]:
# 4.2 product_category_name co the thieu trong bang dich
missing_categories = set(products['product_category_name'].dropna().unique()) - \
                     set(category_translation['product_category_name'])
print('So category trong products nhung KHONG co trong category_translation:', len(missing_categories))
print(sorted(missing_categories)[:10])


In [ ]:
# 4.3 order_payments va order_reviews co N dong / 1 order (fan-out)
pay_per_order = order_payments.groupby('order_id').size()
rev_per_order = order_reviews.groupby('order_id').size()
print('order_payments: max so dong / order =', pay_per_order.max(), '| so order co >1 dong payment =', (pay_per_order > 1).sum())
print('order_reviews : max so dong / order =', rev_per_order.max(), '| so order co >1 dong review  =', (rev_per_order > 1).sum())


In [ ]:
# 4.4 leads_closed.seller_id orphan so voi sellers
orphan_sellers = set(leads_closed['seller_id']) - set(sellers['seller_id'])
print(f'leads_closed co {len(orphan_sellers)}/{leads_closed["seller_id"].nunique()} seller_id KHONG ton tai trong sellers'
      f' ({len(orphan_sellers)/leads_closed["seller_id"].nunique():.1%})')


## 5. Chuẩn hóa zip code (giữ số 0 đầu)

SQLite lưu các cột zip dạng **INTEGER**, nên số 0 ở đầu (vd. `01037`) đã bị mất ngay khi đọc ra (`1037`).
Ép toàn bộ cột zip về chuỗi 5 ký tự (đệm số 0) để so khớp đúng giữa các bảng.

In [ ]:
def normalize_zip(series):
    return series.astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

customers['customer_zip_code_prefix'] = normalize_zip(customers['customer_zip_code_prefix'])
sellers['seller_zip_code_prefix'] = normalize_zip(sellers['seller_zip_code_prefix'])
geolocation['geolocation_zip_code_prefix'] = normalize_zip(geolocation['geolocation_zip_code_prefix'])


## 6. DIM_LOCATION

**Không gộp/aggregate** — giữ nguyên toàn bộ dòng gốc của `geolocation` (mỗi dòng là 1 điểm GPS mẫu),
chỉ thêm `location_id` đánh số thứ tự làm khóa chính (surrogate).

Vì zip_code_prefix không unique, `DIM_CUSTOMER`/`DIM_SELLER` ở bước sau sẽ khớp theo cả 3 cột
`(zip_code_prefix, city, state)` để tìm đúng `location_id` — thay vì chỉ khớp theo zip.

In [ ]:
dim_location = geolocation.copy()
dim_location.insert(0, 'location_id', range(1, len(dim_location) + 1))

print('DIM_LOCATION:', dim_location.shape)
dim_location.head()


## 7. DIM_PRODUCT_CATEGORY

Bổ sung các category có trong `products` nhưng thiếu bản dịch tiếng Anh (tránh mất dữ liệu khi INNER JOIN).

In [ ]:
dim_product_category = category_translation.copy()

if missing_categories:
    extra = pd.DataFrame({
        'product_category_name': sorted(missing_categories),
        'product_category_name_english': np.nan,
    })
    dim_product_category = pd.concat([dim_product_category, extra], ignore_index=True)

print('DIM_PRODUCT_CATEGORY:', dim_product_category.shape)
dim_product_category.tail()


## 8. DIM_PRODUCT

In [ ]:
dim_product = products[[
    'product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght',
    'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
]].copy()

print('DIM_PRODUCT:', dim_product.shape)
dim_product.head()


## 9. DIM_CUSTOMER

Tra `location_id` bằng cách khớp `(zip_code_prefix, city, state)` — so khớp không phân biệt hoa/thường —
với `DIM_LOCATION`. Vì cùng 1 bộ ba này vẫn có thể có nhiều điểm GPS trùng, lấy `location_id` của dòng khớp **đầu tiên**.

In [ ]:
geo_lookup = (
    dim_location
    .assign(
        _city_lc=dim_location['geolocation_city'].str.lower(),
        _state_uc=dim_location['geolocation_state'].str.upper(),
    )
    .drop_duplicates(subset=['geolocation_zip_code_prefix', '_city_lc', '_state_uc'], keep='first')
    [['location_id', 'geolocation_zip_code_prefix', '_city_lc', '_state_uc']]
)

customers['_city_lc'] = customers['customer_city'].str.lower()
customers['_state_uc'] = customers['customer_state'].str.upper()

dim_customer = customers.merge(
    geo_lookup,
    left_on=['customer_zip_code_prefix', '_city_lc', '_state_uc'],
    right_on=['geolocation_zip_code_prefix', '_city_lc', '_state_uc'],
    how='left',
)[['customer_id', 'customer_unique_id', 'location_id']]

missing_geo = dim_customer['location_id'].isna().sum()
print('DIM_CUSTOMER:', dim_customer.shape, '| khong tim duoc location_id:', missing_geo)
dim_customer.head()


## 10. DIM_SELLER

Cùng cách khớp `(zip_code_prefix, city, state)` như DIM_CUSTOMER.

In [ ]:
sellers['_city_lc'] = sellers['seller_city'].str.lower()
sellers['_state_uc'] = sellers['seller_state'].str.upper()

dim_seller = sellers.merge(
    geo_lookup,
    left_on=['seller_zip_code_prefix', '_city_lc', '_state_uc'],
    right_on=['geolocation_zip_code_prefix', '_city_lc', '_state_uc'],
    how='left',
)[['seller_id', 'location_id']]

missing_geo = dim_seller['location_id'].isna().sum()
print('DIM_SELLER:', dim_seller.shape, '| khong tim duoc location_id:', missing_geo)
dim_seller.head()


## 11. DIM_REVIEW

Loại trùng theo `review_id` (nếu có) và parse các cột ngày.

In [ ]:
dim_review = order_reviews.drop_duplicates(subset='review_id').copy()
dim_review['review_creation_date'] = pd.to_datetime(dim_review['review_creation_date'], errors='coerce')
dim_review['review_answer_timestamp'] = pd.to_datetime(dim_review['review_answer_timestamp'], errors='coerce')

dim_review = dim_review[[
    'review_id', 'order_id', 'review_score', 'review_comment_title',
    'review_comment_message', 'review_creation_date', 'review_answer_timestamp',
]]

print('DIM_REVIEW:', dim_review.shape)
dim_review.head()


## 12. FACT_ORDER_ITEMS

Grain: 1 dòng / order item. Lấy `customer_id` từ `orders`.

In [ ]:
fact_order_items = order_items.merge(
    orders[['order_id', 'customer_id']], on='order_id', how='left',
)[['order_id', 'order_item_id', 'customer_id', 'seller_id', 'product_id', 'price', 'freight_value']]

print('FACT_ORDER_ITEMS:', fact_order_items.shape)
fact_order_items.head()


## 13. FACT_ORDER_PAYMENT

Grain: 1 dòng / lần thanh toán (`order_id` + `payment_sequential`) — giữ nguyên, không gộp vào FACT_ORDER_ITEMS vì khác grain.

In [ ]:
fact_order_payment = order_payments[[
    'order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value',
]].copy()

print('FACT_ORDER_PAYMENT:', fact_order_payment.shape)
fact_order_payment.head()


## 14. FACT_SELLER_LEAD

Grain: 1 dòng / lead. `LEFT JOIN leads_qualified <- leads_closed` theo `mql_id` (quan hệ 1–0..1, đã xác nhận không trùng, không mồ côi phía leads_closed).
Thêm cờ `is_won` để đánh dấu lead đã chuyển đổi thành seller.

In [ ]:
fact_seller_lead = leads_qualified.merge(leads_closed, on='mql_id', how='left')
fact_seller_lead['is_won'] = fact_seller_lead['seller_id'].notna()

print('FACT_SELLER_LEAD:', fact_seller_lead.shape)
print('So luong da won:', fact_seller_lead['is_won'].sum(), '/', len(fact_seller_lead))
fact_seller_lead.head()


## 15. Kiểm tra tính toàn vẹn tham chiếu (referential integrity)

Xác nhận các bảng Fact vừa dựng có FK khớp với Dimension tương ứng (trừ trường hợp đã biết trước là orphan).

In [ ]:
def check_fk(fact_df, fact_col, dim_df, dim_col, label):
    orphan = set(fact_df[fact_col].dropna()) - set(dim_df[dim_col].dropna())
    print(f'{label:45s} orphan = {len(orphan):6d} / {fact_df[fact_col].nunique()}')
    return orphan

check_fk(fact_order_items, 'customer_id', dim_customer, 'customer_id', 'FACT_ORDER_ITEMS.customer_id -> DIM_CUSTOMER')
check_fk(fact_order_items, 'seller_id',   dim_seller,   'seller_id',   'FACT_ORDER_ITEMS.seller_id -> DIM_SELLER')
check_fk(fact_order_items, 'product_id',  dim_product,  'product_id',  'FACT_ORDER_ITEMS.product_id -> DIM_PRODUCT')
check_fk(dim_product,  'product_category_name', dim_product_category, 'product_category_name', 'DIM_PRODUCT.product_category_name -> DIM_PRODUCT_CATEGORY')
check_fk(dim_customer, 'location_id', dim_location, 'location_id', 'DIM_CUSTOMER.location_id -> DIM_LOCATION')
check_fk(dim_seller,   'location_id', dim_location, 'location_id', 'DIM_SELLER.location_id -> DIM_LOCATION')
check_fk(fact_order_payment, 'order_id', fact_order_items, 'order_id', 'FACT_ORDER_PAYMENT.order_id -> FACT_ORDER_ITEMS.order_id')
check_fk(dim_review,   'order_id', fact_order_items, 'order_id', 'DIM_REVIEW.order_id -> FACT_ORDER_ITEMS.order_id')
_ = check_fk(fact_seller_lead, 'seller_id', dim_seller, 'seller_id', 'FACT_SELLER_LEAD.seller_id -> DIM_SELLER (biet truoc co orphan)')


**Giải thích 2 orphan phát sinh (không phải lỗi ETL):**

- `FACT_ORDER_PAYMENT.order_id` có ~775 dòng không khớp `FACT_ORDER_ITEMS`, `DIM_REVIEW.order_id` có ~725 dòng tương tự.
- Nguyên nhân: `order_items` chỉ ghi nhận khi sản phẩm **thực sự được xử lý/giao**, còn khách hàng vẫn có thể **đã thanh toán hoặc để lại review** cho đơn hàng bị `canceled`/`unavailable` trước khi có item nào được tạo. Kiểm tra theo `order_status` xác nhận >95% các orphan này rơi vào 2 trạng thái đó — đây là hành vi nghiệp vụ thật, không phải lỗi join.

## 16. Xuất kết quả

Xuất ra `/kaggle/working/processed/` (tab **Output** của Kaggle sẽ tự hiện các file này để tải về hoặc dùng làm Input cho notebook khác).

In [ ]:
outputs = {
    'dim_customer.csv': dim_customer,
    'dim_seller.csv': dim_seller,
    'dim_product.csv': dim_product,
    'dim_product_category.csv': dim_product_category,
    'dim_location.csv': dim_location,
    'dim_review.csv': dim_review,
    'fact_order_items.csv': fact_order_items,
    'fact_order_payment.csv': fact_order_payment,
    'fact_seller_lead.csv': fact_seller_lead,
}

for filename, df in outputs.items():
    df.to_csv(OUT_DIR / filename, index=False)
    print(f'Da xuat {filename:28s} shape={df.shape}')
